# Notebook de Teste para Incorporação de Vídeos

Este notebook isola os posts que contêm `<iframe>` na coluna `Image Featured` para facilitar a verificação da lógica de migração de vídeos para o formato Ricos.

In [72]:
import pandas as pd

# Carregar o arquivo CSV principal
try:
    full_df = pd.read_csv('posts_with_meta_and_alt.csv')
    print('CSV carregado com sucesso!')
except FileNotFoundError:
    print('Erro: Arquivo posts_with_meta_and_alt.csv não encontrado.')
    full_df = pd.DataFrame() # Cria um DataFrame vazio para evitar erros subsequentes

CSV carregado com sucesso!


In [73]:
# Filtrar o DataFrame para encontrar posts que contêm '<iframe>' na coluna 'Content'
if not full_df.empty:
    iframe_posts_df = full_df[full_df['Content'].str.contains('<iframe', na=False)].copy()

    print(f'Encontrados {len(iframe_posts_df)} posts com iframes na coluna Content.')
else:
    iframe_posts_df = pd.DataFrame()  # Cria um DataFrame vazio se o original não foi carregado


Encontrados 53 posts com iframes na coluna Content.


In [74]:
# Exibir o DataFrame filtrado para verificação
if not iframe_posts_df.empty:
    # Mostra o ID, Título e o conteúdo da coluna Image Featured
    display(iframe_posts_df[['ID', 'Title', 'Image Featured']])
else:
    print('Nenhum post com iframe encontrado na coluna Image Featured.')

,ID,Title,Image Featured
7,89,Como emitir uma nota fiscal eletrônica utiliza...,https://otimizeseunegocio.com/wp-content/uploa...
8,94,[Planilha Financeira] Se você cuida bem das su...,https://otimizeseunegocio.com/wp-content/uploa...
9,98,Como instalar o Emissor Gratuito?,https://otimizeseunegocio.com/wp-content/uploa...
10,102,Programa para controle de estoque [GRÁTIS],https://otimizeseunegocio.com/wp-content/uploa...
12,124,6 tendências da Feira do Empreendedor para inv...,https://otimizeseunegocio.com/wp-content/uploa...
15,174,EasyGestor - Sistema de Gestão Gratuito,https://otimizeseunegocio.com/wp-content/uploa...
16,175,Como fazer uma venda e controlar estoque no Ea...,https://otimizeseunegocio.com/wp-content/uploa...
21,241,As 5 coisas que mais consomem o tempo do empre...,https://otimizeseunegocio.com/wp-content/uploa...
32,384,Markup: aprenda tudo sobre como calcular seu p...,https://otimizeseunegocio.com/wp-content/uploa...
34,404,[Passo a passo] Como resolver erro no EasyGestor,https://otimizeseunegocio.com/wp-content/uploa...


### Teste de Conversão para Ricos

In [75]:
# Import the conversion function and other necessary libraries
import sys
import os
import json
import pandas as pd

# Add the 'src' directory to the Python path to allow importing from 'parsers'
# This is a common practice when running notebooks from the project root
project_root = os.path.abspath(os.path.join(os.getcwd()))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.parsers.ricos_parser import convert_html_to_ricos

if not iframe_posts_df.empty:
    # Get the first 3 posts with iframes (or fewer if there aren't 3)
    test_df = iframe_posts_df.head(10)
    print(f"--- Testando a conversão para os primeiros {len(test_df)} posts ---")

    # Iterate over the test posts and convert the HTML to Ricos
    for index, row in test_df.iterrows():
        post_id = row['ID']
        # Use the 'Content' column as per the user's execution of the notebook
        html_content = row['Content'] 
        
        print(f"Testando Post ID: {post_id} ---")
        
        if pd.isna(html_content):
            print("Conteúdo HTML está vazio. Pulando.")
            continue

        # Converte o HTML para Ricos
        ricos_json = convert_html_to_ricos(html_content)
        
        # Imprime o JSON resultante
        print(json.dumps(ricos_json, indent=2))
        
        print("" + "="*50 + "")

else:
    print("Nenhum post com iframe para testar.")

--- Testando a conversão para os primeiros 10 posts ---
Testando Post ID: 89 ---
{
  "nodes": [
    {
      "type": "VIDEO",
      "id": "132f5fc902d5",
      "videoData": {
        "containerData": {
          "width": {
            "size": "CONTENT"
          },
          "alignment": "CENTER"
        },
        "video": {
          "src": {
            "url": "https://www.youtube.com/embed/KdKKGgWAkdM"
          }
        }
      }
    },
    {
      "type": "PARAGRAPH",
      "nodes": [
        {
          "type": "TEXT",
          "textData": {
            "text": "",
            "decorations": []
          }
        }
      ],
      "paragraphData": {
        "textStyle": {
          "textAlignment": "JUSTIFY"
        }
      }
    },
    {
      "type": "PARAGRAPH",
      "nodes": [
        {
          "type": "TEXT",
          "textData": {
            "text": "\n\n\u00a0\n\nTutorial completo com material extra para emitir nota fiscal eletr\u00f4nica usando o programa emissor g

### Teste de Migração para o Wix (Criação de Posts)

In [76]:
# Criar um CSV separado com TODOS os posts que têm iframes para testar
if not iframe_posts_df.empty:
    # Pegar TODOS os posts com iframe (não apenas 3)
    test_posts = iframe_posts_df.copy()
    
    # Salvar em arquivo separado - NÃO substitui o original
    test_csv_path = 'posts_videos_teste.csv'  # Na raiz do projeto
    test_posts.to_csv(test_csv_path, index=False, encoding='utf-8')
    
    print(f"✅ CSV de teste criado: {test_csv_path}")
    print(f"📊 TODOS os posts com iframes: {len(test_posts)}")
    
    print(f"\n🎬 Posts incluídos (primeiros 10):")
    for idx, row in test_posts.head(10).iterrows():
        print(f"  - ID {row['ID']}: {row['Title'][:60]}...")
        print(f"    Slug: {row['Slug']}")
    
    if len(test_posts) > 10:
        print(f"  ... e mais {len(test_posts) - 10} posts")
    
    print(f"\n🚀 Como testar (SEM INTERFERIR no funcionamento atual):")
    print(f"1. Copie o arquivo de teste para docs/:")
    print(f"   cp {test_csv_path} docs/")
    print(f"2. Temporarily mova o CSV principal:")
    print(f"   mv docs/posts_with_meta_and_alt.csv docs/posts_with_meta_and_alt_BACKUP.csv")
    print(f"3. Renomeie o teste para o nome que o main.py procura:")
    print(f"   mv docs/{test_csv_path} docs/posts_with_meta_and_alt.csv")
    print(f"4. Execute: python main.py")
    print(f"5. Depois do teste, restaure o original:")
    print(f"   mv docs/posts_with_meta_and_alt.csv docs/{test_csv_path}")
    print(f"   mv docs/posts_with_meta_and_alt_BACKUP.csv docs/posts_with_meta_and_alt.csv")
    
    print(f"\n💡 O CSV original permanece intocado como backup!")
    print(f"⚠️ ATENÇÃO: Vai migrar {len(test_posts)} posts com vídeos!")
    
else:
    print("❌ Nenhum post com iframe encontrado.")

✅ CSV de teste criado: posts_videos_teste.csv
📊 TODOS os posts com iframes: 53

🎬 Posts incluídos (primeiros 10):
  - ID 89: Como emitir uma nota fiscal eletrônica utilizando o emissor ...
    Slug: como-emitir-uma-nota-fiscal-eletronica-utilizando-o-emissor-gratuito
  - ID 94: [Planilha Financeira] Se você cuida bem das suas finanças va...
    Slug: movimento-de-caixa-planilha-gratis
  - ID 98: Como instalar o Emissor Gratuito?...
    Slug: como-instalar-o-emissor-gratuito
  - ID 102: Programa para controle de estoque [GRÁTIS]...
    Slug: programa-para-controle-de-estoque-gratis
  - ID 124: 6 tendências da Feira do Empreendedor para investir em 2018...
    Slug: 6-tendencias-para-investir-em-2018
  - ID 174: EasyGestor - Sistema de Gestão Gratuito...
    Slug: easygestor-sistema-de-gestao-gratuito
  - ID 175: Como fazer uma venda e controlar estoque no EasyGestor...
    Slug: como-fazer-uma-venda-e-controlar-estoque-no-easygestor
  - ID 241: As 5 coisas que mais consomem o tempo do e

In [77]:
# Função específica para testar conversão de vídeos
import re
from src.parsers.ricos_parser import _extract_video_info, _create_video_node

def test_video_conversion():
    """Testa especificamente a conversão de iframes de vídeo"""
    print("=== TESTE DE CONVERSÃO DE VÍDEOS ===\n")
    
    # Pegar o primeiro post com iframe para teste detalhado
    if not iframe_posts_df.empty:
        test_post = iframe_posts_df.iloc[0]
        html_content = test_post['Content']
        
        print(f"Post ID: {test_post['ID']}")
        print(f"Título: {test_post['Title']}")
        print(f"Slug: {test_post['Slug']}\n")
        
        # Extrair todos os iframes do HTML
        iframe_pattern = re.compile(r'<iframe[^>]*src=["\']([^"\']*)["\'][^>]*>', re.IGNORECASE)
        iframes = iframe_pattern.findall(html_content)
        
        print(f"Encontrados {len(iframes)} iframes:\n")
        
        for i, iframe_src in enumerate(iframes, 1):
            print(f"--- IFRAME {i} ---")
            print(f"URL: {iframe_src}")
            
            # Testar detecção de vídeo
            video_info = _extract_video_info(iframe_src)
            if video_info:
                print(f"✅ Detectado como vídeo: {video_info['platform']} (ID: {video_info['id']})")
                
                # Criar o nó de vídeo
                video_node = _create_video_node(iframe_src)
                print(f"📹 Nó de vídeo criado:")
                print(json.dumps(video_node, indent=2, ensure_ascii=False))
            else:
                print("❌ Não detectado como vídeo suportado")
            print()
        
        # Mostrar o HTML original dos iframes para comparação
        print("--- HTML ORIGINAL DOS IFRAMES ---")
        iframe_full_pattern = re.compile(r'<iframe[^>]*>.*?</iframe>', re.IGNORECASE | re.DOTALL)
        full_iframes = iframe_full_pattern.findall(html_content)
        
        for i, iframe_html in enumerate(full_iframes, 1):
            print(f"IFRAME {i}:")
            print(iframe_html)
            print()
            
        # Teste de conversão completa
        print("--- CONVERSÃO COMPLETA PARA RICOS ---")
        ricos_result = convert_html_to_ricos(html_content)
        
        # Procurar por nós de vídeo no resultado
        video_nodes = []
        def find_video_nodes(nodes):
            for node in nodes:
                if node.get('type') == 'VIDEO':
                    video_nodes.append(node)
                if 'nodes' in node and isinstance(node['nodes'], list):
                    find_video_nodes(node['nodes'])
        
        find_video_nodes(ricos_result.get('nodes', []))
        
        print(f"✅ Encontrados {len(video_nodes)} nós de vídeo no resultado Ricos:")
        for i, video_node in enumerate(video_nodes, 1):
            print(f"\nVÍDEO {i}:")
            print(json.dumps(video_node, indent=2, ensure_ascii=False))
    
    else:
        print("Nenhum post com iframe encontrado para teste.")

# Executar o teste
test_video_conversion()

=== TESTE DE CONVERSÃO DE VÍDEOS ===

Post ID: 89
Título: Como emitir uma nota fiscal eletrônica utilizando o emissor gratuito
Slug: como-emitir-uma-nota-fiscal-eletronica-utilizando-o-emissor-gratuito

Encontrados 1 iframes:

--- IFRAME 1 ---
URL: https://www.youtube.com/embed/KdKKGgWAkdM
✅ Detectado como vídeo: youtube (ID: KdKKGgWAkdM)
📹 Nó de vídeo criado:
{
  "type": "VIDEO",
  "id": "a1e0241e36f5",
  "videoData": {
    "containerData": {
      "width": {
        "size": "CONTENT"
      },
      "alignment": "CENTER"
    },
    "video": {
      "src": {
        "url": "https://www.youtube.com/embed/KdKKGgWAkdM"
      }
    }
  }
}

--- HTML ORIGINAL DOS IFRAMES ---
IFRAME 1:
<iframe src="https://www.youtube.com/embed/KdKKGgWAkdM" width="560" height="315" frameborder="0" allowfullscreen="allowfullscreen"></iframe>

--- CONVERSÃO COMPLETA PARA RICOS ---
✅ Encontrados 1 nós de vídeo no resultado Ricos:

VÍDEO 1:
{
  "type": "VIDEO",
  "id": "e6aae395a535",
  "videoData": {
    "conta

### Teste Específico de Conversão de Vídeos